In [1]:
import os 
from dotenv import load_dotenv
from groq import Groq 
from langchain_community.document_loaders import PyPDFLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings 
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain 
from langchain_groq import ChatGroq
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory


C:\Users\arvin\AppData\Local\Temp\ipykernel_18088\3128173793.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\arvin\OneDrive\Desktop\GAMES AND STUFF\SON\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [3]:
llm = ChatGroq(
    model ="llama-3.3-70b-versatile",
    temperature=0
)

In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name ="BAAI/bge-small-en-v1.5"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5527.41it/s]


In [5]:
loader = PyPDFLoader("TechNova_HR_Policy.pdf")
docs = loader.load()

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 150
)
docs = text_splitter.split_documents(docs)

In [22]:
if not os.path.exists("./chroma_db"):
    print("Creating vector database....")

    vectordb = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        persist_directory="./chroma_db",
        collection_name="hr_policy"
    )
else :
    print("loading existing vector database...")

    vectordb = Chroma(
        persist_directory="./chroma_db",
        embedding_function=embeddings,
        collection_name="hr_policy"
    )

loading existing vector database...


In [23]:
retriever = vectordb.as_retriever(
    search_type = "mmr",
    search_kwargs = {
        "k" : 4,
        "fetch_k" : 30,
        "lambda_mult" : 0.7,
        
    }
)

In [24]:
query = "What is the leave policy?"

retrieved_docs = retriever.invoke(query)

In [25]:
for i, doc in enumerate(retrieved_docs, start =1):
    print(f"Document {i}")
    print("-" * 60)
    print(doc.page_content)
    print()

Document 1
------------------------------------------------------------
Chapter 3: Leave Policy
3.1 Leave Entitlements
All permanent employees are entitled to the following leave per calendar year. Leave is
credited on January 1st each year. New joiners receive leave on a pro-rata basis depending on
their date of joining.
Leave Type
Days Per Year
Carry Forward
Encashable
Earned Leave (EL)
18 days
Up to 30 days
Yes — at exit
Sick Leave (SL)
10 days
No
No
Casual Leave (CL)
8 days
No
No
Maternity Leave
26 weeks
No
No
Paternity Leave
10 days
No
No
Bereavement Leave

Document 2
------------------------------------------------------------
If an employee exhausts all entitled leaves and requires additional time off, they may apply for
Leave Without Pay subject to manager and HR approval. LWP of more than 30 consecutive
days may affect annual appraisal, increments, and benefit calculations for that year.
Note: Public holidays declared by the company are separate from leave entitlements and are

In [26]:
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question",
    "which might reference context in the chat history, "
    "without the chat history"
    "Do NOT answer the question, just reformulate it if needed"
    "and otherwise return it as is."
)
contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human" , "{input}")
    ]
)

In [27]:
history_aware_retriever = create_history_aware_retriever(
    llm,
    retriever,
    contextualize_q_prompt
)

In [28]:
qa_system_prompt = """
You are an AI assistant that answers questions only from the uploaded HR Policy document.

Rules:
1. Answer ONLY using the retrieved context.
2. Never use outside knowledge.
3. If the answer is not explicitly present in the context, reply:
   "I don't know based on the uploaded document."
4. If multiple retrieved passages discuss the same topic, combine them into one consistent answer.
5. Do not produce contradictory statements.
6. Use bullet points whenever appropriate.
7. Keep the answer clear and concise.

Retrieved Context:
{context}
"""

In [29]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", qa_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}")
    ]
)

In [30]:
document_chain = create_stuff_documents_chain(
    llm, 
    qa_prompt
)

In [31]:
rag_chain = create_retrieval_chain(
    history_aware_retriever,
    document_chain
)

In [32]:
store = {}

In [33]:
def get_session_history(session_id : str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [34]:
conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain, 
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer"
)

c:\Users\arvin\OneDrive\Desktop\GAMES AND STUFF\SON\venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [35]:
response = conversational_rag_chain.invoke(
    {"input": "What is the leave policy?"},
    config={
        "configurable": {
            "session_id": "user_1"
        }
    }
)

print(response["answer"])

The leave policy is as follows:
* Leave entitlements are credited on January 1st each year.
* New joiners receive leave on a pro-rata basis depending on their date of joining.
* The types of leave and their entitlements are:
  * Earned Leave (EL): 18 days per year, can be carried forward up to 30 days, and is encashable at exit.
  * Sick Leave (SL): 10 days per year, cannot be carried forward, and is not encashable.
  * Casual Leave (CL): 8 days per year, cannot be carried forward, and is not encashable.
  * Maternity Leave: 26 weeks per year, cannot be carried forward, and is not encashable.
  * Paternity Leave: 10 days per year, cannot be carried forward, and is not encashable.
* If an employee exhausts all entitled leaves, they may apply for Leave Without Pay (LWP) subject to manager and HR approval.
* LWP of more than 30 consecutive days may affect annual appraisal, increments, and benefit calculations for that year.
* Public holidays declared by the company are separate from leave

In [36]:
response = conversational_rag_chain.invoke(
    {"input": "Can employees carry it forward?"},
    config={
        "configurable": {
            "session_id": "user_1"
        }
    }
)

print(response["answer"])

According to the uploaded document, employees can carry forward a maximum of 30 accumulated Earned Leave (EL) days to the next calendar year. Any EL exceeding the carry-forward limit lapses. There is no information about carrying forward other types of leaves, such as Sick Leave.
